In [1]:
from pyspark.sql.functions import col,to_timestamp,regexp_extract,when,regexp_replace,lower,trim,md5,concat_ws

from delta.tables import DeltaTable

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 3, Finished, Available, Finished, False)

In [2]:
jobs = spark.read.format("delta").table("bronze_jobs")

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 4, Finished, Available, Finished, False)

In [3]:
display(jobs.limit(2))

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 06f636ad-10c2-4913-a88a-2281ba4a39b2)

## **clean date**

In [4]:
jobs = jobs.withColumn("job_date",to_timestamp(col("job_date"),"yyyy-MM-dd'T'HH:mm:ss"))


StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 6, Finished, Available, Finished, False)

In [5]:
display(jobs.limit(2))

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c8aab2d5-f6ee-40f8-b087-7ef92846b2c3)

## **Clean Salary**

In [6]:
jobs = jobs.withColumn("min_salary",regexp_extract(col("salary"),r'\$?(\d+)[k]?',1).cast("int")*1000)

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 8, Finished, Available, Finished, False)

In [7]:
display(jobs.limit(2))

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 06fb7228-718d-4d8a-a855-2526aefe60e8)

In [8]:
jobs = jobs.withColumn("max_salary",regexp_replace(col("salary"),r'.*-|[\$Kk\s]','').cast("int")*1000)
display(jobs.limit(2))

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6173a515-0f71-42bb-83d7-c233db7beb71)

In [9]:
jobs = jobs.withColumn("min_salary",when(col("min_salary").isNull(),0).otherwise(col("min_salary")))\
.withColumn("max_salary",when(col("max_salary").isNull(),0).otherwise(col("max_salary")))


display(jobs.limit(2))

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2b8fed61-9eba-43c0-8c21-514b441e1093)

In [10]:
jobs = jobs.withColumn("avg_salary",when((col("min_salary")==0)&(col("max_salary")==0),0).otherwise((col("min_salary")+col("max_salary"))/2))
display(jobs.limit(2))

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 68174e72-0089-4803-8b57-2fcf2dbbfa20)

## **clean string columns**

In [12]:
text_columns=["company_name","job_title","category","job_type","location"]


for column in text_columns:
    jobs = jobs.withColumn(column,trim(lower(col(column))))


display(jobs.limit(5))

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 63c3bc55-f489-434a-ac1e-f67662fdc39c)

In [14]:
df_final = jobs.withColumn("job_id",md5(concat_ws("||",col("company_name"),col("job_title"),col("job_date")))).select(
 col("job_id"),
 col("company_name"),
 col("job_title"),
 col("category"),
 col("job_type"),
 col("job_date"),
 col("location"),
 col("min_salary"),
 col("max_salary"),
 col("avg_salary")  
)

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 16, Finished, Available, Finished, False)

In [15]:
display(df_final.limit(3))

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b59908da-464e-4cae-b065-427330cf0201)

In [16]:
df_final.dropDuplicates()

StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 18, Finished, Available, Finished, False)

DataFrame[job_id: string, company_name: string, job_title: string, category: string, job_type: string, job_date: timestamp, location: string, min_salary: int, max_salary: int, avg_salary: double]

In [17]:
if spark.catalog.tableExists("silver_jobs"):
    old_job = DeltaTable.forName(spark,"silver_jobs").alias("old")
    old_job.merge(
        df_final.alias("final"),
        "old.job_id = final.job_id"
    )\
    .whenNotMatchedInsertAll().execute()
    print("the new data merged with the old table ")
else:
    df_final.write.format("delta").saveAsTable("silver_jobs")
    print("the silver job table created successfully")



StatementMeta(, 28a15993-8626-4fb6-adbe-e193352f8ee6, 19, Finished, Available, Finished, False)

the silver job table created successfully
